# Origin-faithful LoCoMo and LongMemEval benchmark

**Status:** design sections approved; written-spec review pending  
**Beads epic:** `bd-1gyf`  
**Plan:** `b88fc7d8-6f86-4ea7-9e7c-519fe4039fe7`  
**Scope:** replace the methodology described in `crates/spur-graph/BENCHMARKS.md` with an auditable benchmark restarted from the original LoCoMo and LongMemEval sources.

## Decision

Use an **audited dual-contract benchmark**:

1. a retrieval contract that measures whether flat and graph systems retrieve the origin-defined evidence; and
2. an end-to-end QA contract that replays the origin-native reader, prompt, and judge behavior against immutable retrieval artifacts.

The current official datasets are primary. The legacy LongMemEval paper-era release is a lineage audit only. Retrieval and QA results remain separately named; incompatible contracts are never blended into one headline score.

## Goals

- Preserve every source field needed by the original tasks.
- Measure graph value independently from indexing and lexical retrieval.
- Prevent gold leakage before ranking.
- Use origin-native QA scoring as the authoritative quality result.
- Make every denominator, source revision, ranking, prompt, label, and cost reconstructible.
- Permit retrieval-only publication when paid QA cannot run, while making incomplete QA impossible to present as a full result.

## Non-goals

- Reproduce third-party headline numbers under an altered corpus or reader.
- Treat answer-substring presence as QA accuracy.
- Optimize implementation constants in this design phase.
- Claim that solver proofs establish empirical retrieval or QA performance.

The executable solver cells prove only their explicitly encoded, finite policy gates. Dataset validation and benchmark execution still require tests and artifacts.

## Baseline review and origin audit

### Existing benchmark behavior

Self-runs at repository commit `56585d33650f3b1ff35b0412b15f8a1a8093d396` produced:

| Run | Eligible questions | Reported retrieval | Reported “QA coverage” |
|---|---:|---:|---:|
| Existing LoCoMo exact run | 1,536 | Recall@10 = 0.393 | 45.8% |
| Existing LongMemEval exact run | 470 | Recall@10 = 0.865 | 65.0% |
| LongMemEval after preserving multiline content | 470 | unchanged contract | 73.0% |

The fixture suite passed 19 tests with 2 ignored tests, but it checks decomposition and counts rather than benchmark scores.

These figures are diagnostic only. The existing “QA coverage” searches for an answer substring in retrieved text; it is not origin-native QA scoring. The current retriever is a lexical label scan, not graph traversal. LongMemEval materialization truncates multiline content, and both adapters omit task-relevant fields such as roles, dates, speakers, captions, and question dates. Existing external comparisons therefore do not share a controlled experimental contract.

### Pinned origin sources

| Dataset | Authoritative revision | Data checksum |
|---|---|---|
| LoCoMo | Git commit `3eb6f2c585f5e1699204e3c3bdf7adc5c28cb376` | `79fa87e90f04081343b8c8debecb80a9a6842b76a7aa537dc9fdf651ea698ff4` |
| LongMemEval code | Git commit `9e0b455f4ef0e2ab8f2e582289761153549043fc` | — |
| LongMemEval-S-cleaned | Hugging Face revision `98d7416c24c778c2fee6e6f3006e7a073259d48f` | `d6f21ea9d60a0d56f34a05b609c79c88a451d2ae03597821ea3d5a9678c3a442` |

A manifest must pin both revision and byte checksum. A hash mismatch is fatal.

### Audited cohorts

**LoCoMo:** 10 samples, 272 sessions, 5,882 turns, and 1,986 QA records. Category counts are 282 / 321 / 96 / 841 / 446 for categories 1–5. Of 1,982 questions carrying evidence, 9 contain malformed evidence strings. The audited retrieval denominator is therefore **1,973 fully resolved evidence questions**. All **1,986** questions remain in the QA contract. There are 1,226 caption-bearing turns, and 697 scored questions cite at least one caption-bearing evidence turn.

**LongMemEval-S-cleaned:** 500 questions: 470 non-abstention and 30 abstention. The non-abstention records contain 890 answer-session references and 886 `has_answer` turns (832 user, 54 assistant). Fifty-one non-abstention questions have no user-side answer turn. Thirty-two questions disagree between the session-level gold references and sessions induced from `has_answer`; these gold views remain independent. Thirteen source session IDs repeat with identical text but different dates and must remain distinct occurrence-scoped sessions.

Known upstream defects are recorded, never silently repaired.

## Section 1 — Source model and eligibility

### Canonical source contract

Each dataset adapter produces lossless typed records. The canonical representation retains all raw source fields, including:

- conversation/session and turn order;
- speaker or role;
- session date and question date;
- complete multiline content;
- captions and image metadata;
- QA category/type, answer, adversarial answer/options, and original evidence annotations;
- both LongMemEval session-level and turn-level gold signals.

Full text is stored as data, not encoded into Markdown headings or labels. Labels are derived views only. Flat and graph variants consume the same canonical records.

Every session and turn receives an occurrence-scoped internal ID. A source ID is provenance, not identity: repeated LongMemEval source session IDs at different dates remain distinct occurrences.

### Eligibility

- **LoCoMo retrieval:** include a question iff its evidence list is nonempty and every evidence reference resolves unambiguously. Denominator: 1,973.
- **LoCoMo QA:** include all 1,986 questions; category 3 questions without evidence are still valid adversarial QA items.
- **LongMemEval retrieval:** include the 470 non-abstention questions. Score session-level and turn-level gold separately.
- **LongMemEval QA:** include all 500 questions, including 30 abstention questions.

Malformed or unresolved evidence is a validation finding with an explicit eligibility consequence. It is never rewritten without a versioned compatibility transform. Audited and compatibility outputs have different contract IDs and filenames.

In [ ]:
flowchart TD
    SPEC["`@spec SECTION-1-SOURCE-ELIGIBILITY
@type Eligibility = enum[eligible, ineligible]
@input evidence_nonempty: Bool
@input evidence_all_resolved: Bool
@input raw_fields_preserved: Bool
@input internal_ids_unique: Bool
@input silent_repair: Bool
@input contracts_separate: Bool
@output status: Eligibility
@requires LOSSLESS: raw_fields_preserved = true
@requires UNIQUE_OCCURRENCES: internal_ids_unique = true
@requires NO_SILENT_REPAIR: silent_repair = false
@requires NO_CONTRACT_BLENDING: contracts_separate = true`"]
    ELIGIBLE["`@branch ELIGIBLE
@when evidence_nonempty = true and evidence_all_resolved = true
@ensures ELIGIBLE_STATUS: status = eligible`"]
    INELIGIBLE["`@branch INELIGIBLE
@when evidence_nonempty = false or evidence_all_resolved = false
@ensures INELIGIBLE_STATUS: status = ineligible`"]
    CHECK["`@verify S1_CONSISTENT: witness consistency
@verify S1_DETERMINISTIC: prove determinism
@verify S1_COVERED: prove partition_coverage
@verify S1_EXCLUSIVE: prove partition_exclusive
@verify S1_STATUSES: witness each status
@verify S1_ELIGIBLE_WITNESS: witness branch ELIGIBLE
@verify S1_INELIGIBLE_WITNESS: witness branch INELIGIBLE`"]
    SPEC --> ELIGIBLE --> CHECK
    SPEC --> INELIGIBLE --> CHECK

## Section 2 — Retrieval and graph-value evaluation

### Controlled variants

1. **oracle** — origin evidence, used only as a metric and reader upper-bound check.
2. **recent** — deterministic chronological baseline.
3. **flat_bm25** — lexical retrieval over canonical turns/sessions.
4. **graph_index_only** — graph-derived indexed representations without traversal at query time.
5. **graph_traversal** — query-time graph traversal whose terminal results resolve to canonical provenance.

Memory construction completes before the question is revealed. Rankers may see only the allowed query and corpus views. They must not see answers, question types, evidence labels, `has_answer`, or IDs whose names reveal the answer. The non-oracle variants share the same query, corpus, serialization, tie policy, and tokenization contract.

The top-`k` budget counts unique returned **turn or session occurrences**, never intermediate graph nodes. Graph entities must resolve to source provenance before scoring. Session and turn rankings are separate artifacts.

### Metrics

**LoCoMo primary:** macro evidence Recall@5 across eligible questions. Also report Recall@1/10, all-evidence-hit diagnostics, per-category numerator/denominator, and caption-evidence slices.

**LongMemEval primary:** Recall-All@5/10 and NDCG@5/10. Turn-level output additionally reports @50. Recall-Any is diagnostic. Session gold and turn gold are never substituted for one another.

Do not publish a combined LoCoMo–LongMemEval headline. Report build latency, query latency distribution, peak memory, index size, and reader context tokens beside quality.

The causal graph-value comparisons are:

- `graph_traversal − graph_index_only`: value of traversal;
- `graph_index_only − flat_bm25`: value of graph-derived indexing/representation;
- each against `recent` and `oracle`: lower and upper controls.

In [ ]:
flowchart TD
    SPEC["`@spec SECTION-2-RETRIEVAL-ADMISSION
@type Admission = enum[admitted, rejected]
@input same_query: Bool
@input same_corpus: Bool
@input same_serialization: Bool
@input gold_visible: Bool
@input provenance_resolved: Bool
@input unique_k_budget: Bool
@input separate_dataset_metrics: Bool
@input combined_headline: Bool
@output status: Admission`"]
    ADMITTED["`@branch ADMITTED
@when same_query = true and same_corpus = true and same_serialization = true and gold_visible = false and provenance_resolved = true and unique_k_budget = true and separate_dataset_metrics = true and combined_headline = false
@ensures ADMITTED_STATUS: status = admitted`"]
    REJECTED["`@branch REJECTED
@when same_query = false or same_corpus = false or same_serialization = false or gold_visible = true or provenance_resolved = false or unique_k_budget = false or separate_dataset_metrics = false or combined_headline = true
@ensures REJECTED_STATUS: status = rejected`"]
    CHECK["`@verify S2_CONSISTENT: witness consistency
@verify S2_DETERMINISTIC: prove determinism
@verify S2_COVERED: prove partition_coverage
@verify S2_EXCLUSIVE: prove partition_exclusive
@verify S2_STATUSES: witness each status
@verify S2_ADMITTED_WITNESS: witness branch ADMITTED
@verify S2_REJECTED_WITNESS: witness branch REJECTED`"]
    SPEC --> ADMITTED --> CHECK
    SPEC --> REJECTED --> CHECK

## Section 3 — End-to-end QA

QA consumes immutable ranking JSONL produced by Section 2. It may truncate to a declared `k`, but it may not rerun, reorder, or enrich retrieval.

### LoCoMo track

Replay the origin prompt structure with dates, speakers, captions, temporal instructions, and adversarial options.

- Category 1: original multi-answer F1.
- Categories 2–4: original stemmed token F1.
- Category 5: binary abstention/adversarial decision.
- The audited compatibility shim maps the released `adversarial_answer` field to the false option expected by the origin evaluator; option order uses a deterministic recorded seed.

Report overall, each category, adversarial versus non-adversarial, and retrieval variant.

### LongMemEval track

Serialize chronological JSON history with both user and assistant roles, session dates, and question date. Use the origin extract-then-reason structure. Pin the reader and judge to `gpt-4o-2024-08-06` unless an explicitly named compatibility track pins a different model.

Report micro accuracy, task-macro accuracy, every question type, abstention, and retrieval variant. Cache the hypothesis and the complete judge input/output.

### Completion and statistics

The same reader contract is used for oracle, BM25, graph-index-only, and graph-traversal comparisons. Cache prompt, hypothesis, judge input, label, token counts, cost, model version, prompt hash, and retrieval hash per question.

Paid evaluation requires an explicit flag and cost ceiling. Missing credentials or an interrupted API run yields `qa_pending`; no denominator is dropped, and no label is fabricated. Resume by question ID. Aggregate only after every eligible QA record has a terminal label.

Use paired 95% bootstrap intervals for deltas. For LongMemEval, also report paired binary disagreements. Smoke subsets are labeled smoke and never promoted to headline results. Answer-substring “coverage” is removed from the authoritative contract.

In [ ]:
flowchart TD
    SPEC["`@spec SECTION-3-QA-COMPLETION
@type QaState = enum[qa_complete, qa_pending]
@input retrieval_frozen: Bool
@input complete_denominator: Bool
@input prompt_hashed: Bool
@input model_pinned: Bool
@input credentials_available: Bool
@input longmem_track: Bool
@input judge_complete: Bool
@output status: QaState`"]
    COMPLETE["`@branch COMPLETE
@when retrieval_frozen = true and complete_denominator = true and prompt_hashed = true and model_pinned = true and credentials_available = true and (longmem_track = false or judge_complete = true)
@ensures COMPLETE_STATUS: status = qa_complete`"]
    PENDING["`@branch PENDING
@when retrieval_frozen = false or complete_denominator = false or prompt_hashed = false or model_pinned = false or credentials_available = false or (longmem_track = true and judge_complete = false)
@ensures PENDING_STATUS: status = qa_pending`"]
    CHECK["`@verify S3_CONSISTENT: witness consistency
@verify S3_DETERMINISTIC: prove determinism
@verify S3_COVERED: prove partition_coverage
@verify S3_EXCLUSIVE: prove partition_exclusive
@verify S3_STATUSES: witness each status
@verify S3_COMPLETE_WITNESS: witness branch COMPLETE
@verify S3_PENDING_WITNESS: witness branch PENDING`"]
    SPEC --> COMPLETE --> CHECK
    SPEC --> PENDING --> CHECK

## Section 4 — Validation, artifacts, and publication lifecycle

### Artifact contract

Each run writes:

```text
results/<run-id>/
  manifest.json
  validation.json
  rankings/<variant>.jsonl
  metrics/<dataset>-<granularity>.json
  qa/prompts/
  qa/hypotheses/
  qa/judge-inputs/
  qa/labels/
  report.md
  SHA256SUMS
```

The manifest records repository revision and dirty state; dataset URLs, revisions, and hashes; contract IDs; variant configuration; deterministic seeds; model and prompt versions; timestamps; hardware; and command line. Every metric file includes exact numerator, denominator, exclusions, and source ranking hash.

Fatal validation errors are: source hash mismatch, schema mismatch, duplicate internal IDs, unknown ranked IDs, gold leakage, wrong denominator, or non-finite metrics. Known source defects are nonfatal only when listed in `validation.json` with their eligibility effect.

### Lifecycle

```text
Validated → RetrievalComplete → PublishedRetrieval
                            └→ QaPending
                               └→ QaComplete → PublishedFull
```

`QaPending → PublishedFull` is illegal. Retrieval publication requires valid source hashes, schema, identifiers, leakage checks, denominators, finite metrics, and complete rankings. Full publication requires those gates plus complete QA.

If an upstream compatibility pipeline cannot execute at its pinned revision, record `not_runnable`; do not invisibly repair it.

### Verification matrix

- Unit fixtures: multiline text, captions, speakers, dates, repeated source session IDs, malformed evidence, assistant-side evidence, abstention.
- Upstream golden checks: origin evaluator fixtures and prompt serialization.
- Properties: metrics in `[0,1]`; Recall@`k` monotonic; Recall-All ≤ Recall-Any; deterministic ties; oracle retrieves every valid gold item.
- Leakage tests: each forbidden gold field independently fails admission.
- Full-data gates: pinned hashes and audited counts/denominators.
- Resume test: interrupted QA restarts by question ID without changing completed labels or rankings.

In [ ]:
flowchart TD
    SPEC["`@spec SECTION-4-RELEASE-GATE
@type ReleaseState = enum[blocked, published_retrieval, published_full]
@input source_hashes_valid: Bool
@input schema_valid: Bool
@input ids_valid: Bool
@input gold_leak_free: Bool
@input denominators_valid: Bool
@input metrics_finite: Bool
@input retrieval_complete: Bool
@input qa_complete: Bool
@output status: ReleaseState`"]
    FULL["`@branch FULL
@when source_hashes_valid = true and schema_valid = true and ids_valid = true and gold_leak_free = true and denominators_valid = true and metrics_finite = true and retrieval_complete = true and qa_complete = true
@ensures FULL_STATUS: status = published_full`"]
    RETRIEVAL["`@branch RETRIEVAL
@when source_hashes_valid = true and schema_valid = true and ids_valid = true and gold_leak_free = true and denominators_valid = true and metrics_finite = true and retrieval_complete = true and qa_complete = false
@ensures RETRIEVAL_STATUS: status = published_retrieval`"]
    BLOCKED["`@branch BLOCKED
@when source_hashes_valid = false or schema_valid = false or ids_valid = false or gold_leak_free = false or denominators_valid = false or metrics_finite = false or retrieval_complete = false
@ensures BLOCKED_STATUS: status = blocked`"]
    CHECK["`@verify S4_CONSISTENT: witness consistency
@verify S4_DETERMINISTIC: prove determinism
@verify S4_COVERED: prove partition_coverage
@verify S4_EXCLUSIVE: prove partition_exclusive
@verify S4_STATUSES: witness each status
@verify S4_FULL_WITNESS: witness branch FULL
@verify S4_RETRIEVAL_WITNESS: witness branch RETRIEVAL
@verify S4_BLOCKED_WITNESS: witness branch BLOCKED`"]
    SPEC --> FULL --> CHECK
    SPEC --> RETRIEVAL --> CHECK
    SPEC --> BLOCKED --> CHECK

## Implementation boundaries and acceptance criteria

Implementation planning should isolate these components:

1. source adapters and canonical record schema;
2. validation and eligibility reporting;
3. ranker interface plus oracle, recent, and BM25 baselines;
4. graph-index-only and graph-traversal variants;
5. retrieval metrics, efficiency telemetry, and immutable ranking artifacts;
6. origin-native QA replay, cache, resume, and cost control;
7. report generation and replacement of `BENCHMARKS.md`.

A first release is acceptable only when:

- pinned full-data validation reproduces the audited cohort counts;
- all five rankers run through one shared interface with leakage checks;
- rankings are deterministic and independently replayable;
- retrieval metrics pass golden and property tests;
- missing API credentials produce a valid retrieval publication marked `qa_pending`;
- a credentialed full run produces complete origin-native QA artifacts before `published_full`;
- the report clearly separates audited results, compatibility results, and smoke results.

Open implementation choices such as BM25 library, graph traversal algorithm, caching backend, and bootstrap sample count must be decided in the implementation plan with tests or measurements. They are not silently fixed by this design.

## Solver evidence register

All formal cells use registry profile `relational_lia`, profile version 1, with the QF_LIA Bool/Int/enum theory. Each cell was statically checked, executed, and reread after execution. All mandatory proof facets report `matched`; all solver evidence is fresh against the displayed source.

| Section | Cell ID | Source hash | IR hash | Report hash | Obligations | Result |
|---|---|---|---|---|---:|---|
| 1 — source eligibility | `4a894784-6acc-4577-a020-2bc52afc4354` | `f0565008fcefbbf5640fbec786c4e2b40d2744f4f93108277e15becbd0e45b5b` | `7deb025362cd2b975cad6d4d99e2d3dd9fbf651e5c96651bf7018edfafdc97cd` | `e4a5e516f88451423bd906712ab0c66736d70d1d8a683d6fec70044d2dca91da` | 8 | verified |
| 2 — retrieval admission | `44a9f3f1-1cec-47f2-9bfd-70d47aa4bce8` | `128eb4f35bbfa06f711bf4ce1487756ce28ed719a1f40181d99f6e0cf9680aed` | `6fdb3756070d87e4329744dc723a013043a8064c8b5775d0c6930525b6e0d82c` | `283b2debeb80f79d471630b24fd819afd4401e2df9d4db90d068efe359bd281d` | 8 | verified |
| 3 — QA completion | `05ba89a3-86e8-4598-a824-0f82e70e01a6` | `09e03105eb8e477038b71912cc195a8c68106fad67ee0a93485ac33d42cd1aa6` | `846f4dbc694dc277db27541ae8709cdca2513d8e868838e4b9c0f63b6f3b77f4` | `a42278f977a0a8882861d561c824c888fc5694b9c4c555a49da71d9bbc665220` | 8 | verified |
| 4 — release gate | `3bd36132-3268-4f85-b93c-172c52af6f54` | `a655c8a5a41f83208ae924d0f86cada1a1e002c703c8485c7843a1b5fbc8c7b7` | `2ed76d4818dade90485eae41728c431d051c35e2750e4de717b2dc72e4499533` | `e81e19010e7f2bba0046360ef3c94736639f6474c661bda3663ef029731ac008` | 10 | verified |

The in-notebook proofs are reinforced by independently persisted solver runs made while reviewing each section:

- Cohort arithmetic: `sol_cf83e622b2764c47` using `data_integrity.aggregate_balance`.
- Section 1 counterexample search: `sol_e0e95344ac514b0b` — unsatisfiable.
- Section 2 counterexample search: `sol_aad4dc6d2bfb4160` — unsatisfiable.
- Section 3 counterexample search: `sol_f2e1175ca1c1420e` — unsatisfiable.
- Section 4 release counterexample search: `sol_62945f70529248ce` — unsatisfiable.
- Section 4 positive workflow paths: `sol_f02310a2f4a341d1` — pass.
- Illegal `QaPending → PublishedFull` path: `sol_bf17ffd70485452a` — correctly rejected.

### Interpretation boundary

These results prove the exact encoded finite policy:

- each decision relation has a satisfiable model;
- every declared status is reachable;
- branches cover the admitted input space without overlap;
- a single input cannot produce conflicting statuses;
- full publication cannot be selected unless QA is complete.

They do **not** prove that upstream data are correct, that the implementation preserves every field, that gold never leaks through an unmodeled channel, or that graph traversal improves retrieval. Those claims require the validation suite and benchmark runs specified above.